[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/convection/convection.ipynb)

# Rayleigh–Bénard Convection

This example computes Rayleigh–Bénard convection in a layer of liquid between a warm floor and a cool ceiling. While the liquid stays still it conducts heat up, and once buoyancy beats viscosity and thermal conduction it turns over in rolls. Warm liquid is lighter, so the still layer is top-heavy; a rising parcel keeps its buoyancy only if it rises before its warmth conducts away, and the Rayleigh number compares the two times. Starting from a slightly perturbed conductive profile, the layer converts potential energy into the kinetic energy of rolls that viscosity dissipates as heat ([Bergé and Dubois, 1984](https://doi.org/10.1080/00107518408210730)).

The algebra is the projective plane `x+ y+ w0` joined with three degenerate directions for the quantities (`x+ y+ w0 v0 m0 s0`), namely volume, mass and entropy. A carrier's contents are modelled as its parcel, `volume + (density - expansion * warmth) * mass + warmth * entropy`: a unit of volume `v.dual()` has less mass when it is warmer and has more entropy. A drive `-gradient ^ potentials` is the fall of the potentials, minus their gradient wedged with them, and pairs with the flux of each quantity. Gravity's drive, minus the geopotential's gradient wedged with mass, pulls on the advected mass, while the temperature's drive, wedged with entropy, maps through a conductivity to the conductive entropy flux. Flow is modelled as a stream function on the vertices while pressure keeps each dual cell incompressible and drops out of the work, and momentum advection is omitted at high Prandtl number. Gravity's drive pairs with `flows & parcels` to give the buoyant work on the flow and with `places & parcels` to give minus the potential energy, so one pairing exchanges potential and kinetic energy without loss.

In vector-calculus notation, the high-Prandtl Oberbeck–Boussinesq equations couple the Stokes balance $-\nabla p/\rho_0 + \nu\nabla^2\mathbf v + g\alpha T\,\hat{\mathbf z} = \mathbf 0$ and $\nabla\cdot\mathbf v = 0$ to advection–diffusion $\partial_t T + \mathbf v\cdot\nabla T = \kappa\nabla^2 T$, eliminating $p$ through the biharmonic streamfunction equation $\nu\nabla^4\psi = -g\alpha\,\partial_x T$ with $\mathbf v = (\partial_z\psi, -\partial_x\psi)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Callable, Iterator
from itertools import islice

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import Sparse, SparseExtensor, spdiag
from examples.util.animation import show_states
from examples.fluids.convection import render, scenarios
from examples.fluids.convection.scenarios import layer

# The projective plane x+ y+ w0 joined with three degenerate directions for volume, mass and entropy.
plane = Algebra("x+y+")
space = Algebra("x+y+w0")
quantities = Algebra("v0m0s0")
ga = space * quantities
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
# Planes in the spatial sector that read horizontal and vertical offsets without w.
Upright = ga.gatype.vector().derive.within(plane)                             # Vector+
Drive = ga.gatype.bivector().derive.touching(plane, quantities)               # Bivector+
Flux = ga.gatype.antibivector()                                               # Antibivector-
# Reference origin at unit w weight, horizontal and vertical directions, and the spatial planes that read them.
origin = mv.w.dual()                                                          # [] Antivector-
length = mv.x.dual()                                                          # [] Antivector-
height = mv.y.dual()                                                          # [] Antivector-
directions = stack((length, height))                                          # [2] Antivector-
readings = stack((mv.x, mv.y))                                                # [2] Vector+
# Units of the three quantities: volume, mass and entropy.
volume = mv.v.dual()                                                          # [] Antivector-
mass = mv.m.dual()                                                            # [] Antivector-
entropy = mv.s.dual()                                                         # [] Antivector-

## The layer

The domain is a triangulated strip one depth high, closed periodically along its length so that it has a floor and a ceiling without side walls. Triangle geometry is evaluated on the same strip opened at its seam, where the last column repeats the first one period further along, while fields live on the closed strip's vertices and map onto the opened strip for rendering. Lengths are measured in layer depths, times in thermal diffusion times across the layer, and temperatures in floor-to-ceiling differences.

In [ ]:
# Periodic closed strip and opened strip four depths long, vertex-reading maps between them, and vertex points.
closed, opened, closing, opening, places = layer(scenarios.n_along, scenarios.n_across, scenarios.period)
vertices = (mv(Upright, places) + mv.w).dual().field()                        # Antivector-[V_opened]
render.draw_layer(opened, vertices, scenarios.period);

The triangulated strip lies between its warm floor in red and its cool ceiling in blue.

## Convection from a still layer

A stream function on the vertices represents the incompressible flow: a unit rise at one corner of a triangle drives fluid along the facing side. Kinetic energy pairs inertial momentum with velocity, while thermal conduction gathers the entropy flux driven by the corner simplex gradients, `s & (gradients ^ conductivity(drives))`, onto the vertices. Gravity's drive performs buoyant work by pulling on the mass in each advected parcel. The liquid has water's Prandtl number of seven and starts from a perturbed conductive profile. The cells below take a few seconds, most of it stepping the liquid and rendering the animation.

In [ ]:
# Reference density, dimensionless viscosity (the Prandtl number), thermal expansion per degree, and gravity from the Rayleigh number.
density, viscosity, expansion = scenarios.density, scenarios.prandtl, scenarios.expansion
gravity = scenarios.rayleigh * viscosity / expansion
# Absolute reference temperature near 300 K and volumetric heat capacity in flow-energy units.
reference_temperature, heat_capacity = scenarios.reference_temperature, scenarios.heat_capacity
# Time step, total steps, frame stride, and initial steps run before rendering.
dt, n_steps, n_stride, n_unshown = scenarios.dt, scenarios.n_steps, scenarios.n_stride, scenarios.n_unshown

In [ ]:
# Triangle geometry evaluated on the opened strip and assembled onto the closed strip's vertices.
corners_open = opened.incidence[2, 0] * vertices                              # [3] Antivector-[F]
facing = opened.off[2, 0] * vertices                                          # [2, 3] Antivector-[F]
facing_lines = facing[0] & facing[1]                                          # [3] Antibivector-[F]
unread = volume & mass & entropy                                              # [] Antitrivector-
# Twice each triangle's signed area: three corners joined with unit volume, mass and entropy.
double_areas = corners_open[0] & corners_open[1] & corners_open[2] & unread   # Scalar-[F]
areas = double_areas.norm() / 2                                               # Scalar+[F]
# Spatial corner gradients: the plane through the facing line and the quantity directions, normalized at the corner.
gradients = ((facing_lines & unread) / (facing_lines & unread & corners_open)).cast(Upright)   # [3] Vector+[F]
corners = closed.incidence[2, 0]                                              # [3] Scalar+[F <- V]
# Each vertex's dual cell area, a third of each incident triangle, and the corner-averaging map onto faces.
dual_areas = (corners.adjugate() * (areas / 3)).sum(axis=0)                   # Scalar+[V]
corner_mean = corners.mean(axis=0)                                            # Scalar+[F <- V]
# A unit stream function at a corner drives flow along the facing side, giving each corner's integrated flow and velocity.
sides = opened.boundary[2, 1] * (opened.d[0] * vertices)                      # [3] Antivector-[F]
flows = sides / 2                                                             # [3] Antivector-[F]
velocities = flows / areas                                                    # [3] Antivector-[F]
# Stream-function velocity on each face, and the outflow from each dual cell gathered by wedging the corner gradients with a flux.
stream_velocity = (spdiag(velocities * Scalar) * corners).sum(axis=0)         # (Antivector- <- Scalar+)[F <- V]
cell_outflow = (corners.adjugate() * spdiag(areas * (gradients ^ Flux))).sum(axis=0)    # (Antivector- <- Antibivector-)[V <- F]

In [ ]:
# Kinetic form: inertia pairs corner velocities across each triangle and gathers onto the vertices.
inertia = density * (readings * (readings & Point)).sum(axis=0)               # [] Vector+ <- Antivector-
local = areas * (inertia(velocities[:, None]) & velocities[None, :])          # [3, 3] Scalar+[F]
kinetic = (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[V <- V]
# Mass-lumped vorticity on dual cells, and the viscous form from its cell-weighted square.
vorticity = spdiag(1 / (density * dual_areas)) * kinetic                      # Scalar+[V <- V]
viscous = viscosity * ~vorticity * spdiag(dual_areas) * vorticity             # Scalar+[V <- V]
# Thermal drives at the corners, minus each simplex gradient wedged with entropy, and isotropic entropy conductivity.
drives = -gradients ^ mv.s                                                    # [3] Bivector+[F]
conductivity = ((directions & entropy) * (Drive & (directions & entropy))).sum(axis=0)   # [] Antibivector- <- Bivector+
# Conductive entropy outflow: corner gradients wedged with the conductive flux and gathered onto the vertices.
local = areas * (mv.s & (gradients[:, None] ^ conductivity(drives[None, :]))) # [3, 3] Scalar+[F]
conduction = (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[V <- V]
# Gravity's drive, and the gravitational potential energy change per unit warmth at each vertex.
pull = -gravity * mv.ym                                                       # [] Bivector+
places = closing * vertices                                                   # Antivector-[V]
lift = pull & (places & (entropy - expansion * mass))                         # Scalar+[V]

In [ ]:
def holding(form: SparseExtensor, interior: Sparse) -> Callable[[Scalar, Scalar], Scalar]:
    """A vertex operator solved with fixed wall values: for a right-hand side and wall values,
    it solves on the free vertices and leaves the wall values unchanged."""
    free = (interior * form * interior.adjugate()) * Scalar                   # (Scalar+ <- Scalar+)[V_free <- V_free]
    return lambda rhs, walls: walls + interior.adjugate() * free.solve(interior * (rhs - form * walls))


# Implicit midpoint solvers for vorticity-stream momentum and heat conduction with held walls.
interior = closed.reading(closed.interior[0], 0)                              # Scalar+[V_free <- V]
held = interior.adjugate() * interior                                         # Scalar+[V <- V]
flow_step = holding(kinetic / (dt / 2) + viscous, interior)
heat_step = holding(dual_areas / (dt / 2) + conduction, interior)
# Perturbed conductive initial temperature and fixed wall temperatures.
start = scenarios.conducting(closing * (mv.y & vertices), scenarios.disturbance)   # Scalar+[V]
wall_temperature = start - held * start                                       # Scalar+[V]

In [ ]:
def convecting(temperature: Scalar) -> Iterator[tuple[Scalar, Scalar, tuple[Scalar, Scalar, Scalar]]]:
    """Stream function and temperature at regular intervals from rest, with cumulative viscous
    dissipation, conductive potential-energy transfer, and entropy production, each a scalar."""
    # Initial rest state, initial midpoint temperature, and zeroed cumulative budget totals.
    stream = 0 * temperature                                                  # Scalar+[V]
    middle_temperature = temperature                                          # Scalar+[V]
    dissipated = conducted = produced = 0 * temperature.sites.sum()           # [] Scalar+
    for step in range(n_steps + 1):
        if step % n_stride == 0:
            yield stream, temperature, (dissipated, conducted, produced)
        # Extrapolate the midpoint temperature from the previous step's midpoint.
        middle_temperature = 2 * temperature - middle_temperature
        # Midpoint face parcels: unit volume, thermally expanded mass, and entropy.
        warmth = corner_mean * middle_temperature                             # Scalar+[F]
        parcels = volume + (density - expansion * warmth) * mass + warmth * entropy   # Antivector-[F]
        # Buoyant work on each corner's flow drives the stream function to the step's midpoint.
        load = (corners.adjugate() * (pull & (flows & parcels[..., None]))).sum(axis=-1)   # Scalar+[V]
        middle_stream = flow_step(kinetic / (dt / 2) * stream + load, 0 * stream)   # Scalar+[V]
        # Midpoint flow advects face entropy; cell outflow and thermal conduction advance temperature to the midpoint.
        velocity = stream_velocity(middle_stream)                             # Antivector-[F]
        flux = velocity & (warmth * entropy)                                  # Antibivector-[F]
        advected = mv.s & cell_outflow(flux)                                  # Scalar+[V]
        middle_temperature = heat_step(dual_areas / (dt / 2) * temperature - advected, wall_temperature)   # Scalar+[V]
        # Midpoint budget increments: viscous dissipation, conductive potential-energy transfer, and entropy production.
        dissipation = dt * (middle_stream * (viscous * middle_stream)).sites.sum()   # [] Scalar+
        exchange = held * (conduction * middle_temperature + advected) - advected   # Scalar+[V]
        dissipated = dissipated + dissipation                                 # [] Scalar+
        conducted = conducted + dt * (lift * exchange).sites.sum()            # [] Scalar+
        produced = produced + dissipation / reference_temperature + dt * heat_capacity * (
            middle_temperature * (conduction * middle_temperature)).sites.sum() / reference_temperature ** 2
        # Complete the full step symmetrically past the midpoint.
        stream, temperature = 2 * middle_stream - stream, 2 * middle_temperature - temperature

In [ ]:
# Render each frame once the rolls emerge, and retain the yielded states for the diagnostics.
states = show_states(islice(convecting(start), n_unshown // n_stride, None), render.animate, opened, vertices, opening,
                     scenarios.period, name="convection", duration_ms=scenarios.frame_duration_ms,
                     n_frames=(n_steps - n_unshown) // n_stride + 1, colors=scenarios.n_animation_colours)

Temperature is coloured red above the mean and blue below, with black streamlines spaced evenly in the stream function so that closer lines mark faster flow. The animation begins as rolls emerge in the streamlines while the layer still conducts. Warm plumes rise and cool plumes sink between them, overshooting as they mix the interior and spend the initial top-heaviness. Conduction rebuilds the thermal boundary layers at the floor and ceiling, and after a few oscillations the rolls carry heat steadily across a nearly uniform interior.

## Growth of the flow

Evaluated on the rendered states, the integrated squared stream function rises linearly on a logarithmic scale during exponential roll growth, overshoots as the layer overturns, and settles through damped oscillations.

In [ ]:
# Integrated squared stream function over the layer at each rendered state.
sizes = stack(tuple((dual_areas * stream.squared()).sites.sum() for stream, _, _ in states))   # [n_frames] Scalar+
times = (n_unshown + n_stride * np.arange(len(states))) * dt
render.draw_turnover(times, sizes);

## Energy budget

As the layer overturns, gravitational potential energy converts into roll kinetic energy and viscous dissipation, while thermal conduction and heat exchange with the held walls also shift potential energy. Buoyant work on the flow and the potential energy released by advection cancel whenever both evaluate at the same midpoint flow and temperature. Stepping symmetrically across that midpoint changes the kinetic energy by the exact midpoint work, closing the energy budget to floating-point round-off at every step, while positive viscous and conductive operators keep dissipation and entropy production non-decreasing.

In vector-calculus notation, the exchange corresponds to $\int g\alpha T\,v_z\,dA = -\int g\alpha z\,\mathbf v\cdot\nabla T\,dA$ for an incompressible flow with $\nabla\cdot\mathbf v = 0$ tangential to the walls, and entropy is produced at the rate $\int \big(\rho_0\nu|\boldsymbol\omega|^2/T_0 + \rho_0 c_p\kappa|\nabla T|^2/T_0^2\big)\,dA$.

In [ ]:
# Stream functions, temperatures, and cumulative budget terms across the rendered states.
drawn_streams = stack(tuple(stream for stream, _, _ in states))               # [n_frames] Scalar+[V]
drawn_temperatures = stack(tuple(temperature for _, temperature, _ in states))   # [n_frames] Scalar+[V]
dissipated, conducted, produced = map(stack, zip(*(totals for _, _, totals in states)))   # [n_frames] Scalar+, Scalar+, Scalar+
# Kinetic energy of the flow, and gravitational potential energy of the vertex parcels.
kinetic_energies = (drawn_streams * (kinetic * drawn_streams)).sites.sum() / 2   # [n_frames] Scalar+
vertex_parcels = (volume + (density - expansion * drawn_temperatures) * mass
                  + drawn_temperatures * entropy)                             # [n_frames] Antivector-[V]
potential_energies = -(dual_areas * (pull & (places & vertex_parcels))).sites.sum()   # [n_frames] Scalar+
# Total accounted energy and normalized conservation residual relative to the first rendered state.
accounted = kinetic_energies + potential_energies + dissipated - conducted   # [n_frames] Scalar+
unaccounted = (accounted - accounted[0]) / (dissipated[-1] - dissipated[0])  # [n_frames] Scalar+
render.draw_budget(times, kinetic_energies - kinetic_energies[0], potential_energies - potential_energies[0],
                   dissipated - dissipated[0], conducted - conducted[0], unaccounted, produced - produced[0]);

The top panel plots changes from the first rendered state in kinetic energy in blue, potential energy in red, viscous dissipation in grey, and minus the conductive potential-energy transfer in orange; the four sum to zero throughout. Potential energy drops as the rolls overturn the layer, and once steady convection sets in the boundary heat flux replenishes the potential energy that viscosity dissipates. The middle panel shows the normalized conservation residual below `1e-12`, at floating-point round-off. The bottom panel shows cumulative entropy production rising monotonically, steepening once thin boundary layers form at the floor and ceiling.

In [ ]:
# checks
# Gravity pulls a parcel's mass downward and leaves its volume and entropy untouched.
pull_on_parcel = mv.y & (-mv.ym).commutator(mass + volume + entropy)
assert ((pull_on_parcel + 1).norm() < 1e-12).all()
# After turnover, the interior half of the layer is nearly isothermal on average compared with the 0.5 conductive drop.
layer_heights = mv.y & vertices.batch()
temperatures = (opening * states[-1][1]).batch()
middle_drop = temperatures[(layer_heights - 0.25).norm() < 1e-12].mean() - temperatures[(layer_heights - 0.75).norm() < 1e-12].mean()
assert (middle_drop.norm() < 0.05).all()
# The energy budget closes to floating-point round-off, and cumulative entropy production never decreases.
assert (unaccounted.norm() < 1e-10).all()
assert ((produced[1:] - produced[:-1]) >= 0).all()